# LLM Benchmarking — Google Colab (T4 GPU)
### RV College of Engineering | AI364TA | Cloud Computing Project

**Models:** LLaMA 3.2-3B · Falcon-7B · Mistral-7B  
**Tasks:** Summarization (ROUGE) · Translation (BLEU) · Reasoning (Accuracy)  
**Output:** Results pushed to HuggingFace Datasets Hub → displayed on HF Spaces dashboard

---
**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Paste your HF token and W&B API key in Cell 2
3. Runtime → Run All

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers torch accelerate evaluate rouge-score sacrebleu datasets wandb huggingface_hub sentencepiece bitsandbytes

In [ ]:
# Cell 2 — Login to HuggingFace and Weights & Biases
# PASTE YOUR NEW TOKENS BELOW (never commit this notebook with tokens filled in)
HF_TOKEN = ""       # paste your HuggingFace Write token here
WANDB_API_KEY = "" # paste your W&B API key here

from huggingface_hub import login
import wandb

login(token=HF_TOKEN)
wandb.login(key=WANDB_API_KEY)
print("Logged in to HuggingFace and W&B successfully")

In [ ]:
# Cell 3 — Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — switch to T4 GPU runtime'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Cell 4 — Benchmark test inputs (3 tasks × 5 examples each)

SUMMARIZATION_INPUTS = [
    {
        "prompt": "Summarize the following text in 2-3 sentences:\n\nArtificial intelligence has transformed numerous industries over the past decade. In healthcare, AI systems can now diagnose diseases with accuracy comparable to trained physicians, analyze medical images to detect early-stage cancers, and predict patient outcomes based on electronic health records. The technology processes vast amounts of medical data far faster than humans, identifying patterns that might otherwise go unnoticed. However, concerns remain about data privacy, algorithmic bias, and the need for regulatory oversight to ensure these systems are safe and equitable for all patients.\n\nSummary:",
        "reference": "AI has transformed healthcare by enabling disease diagnosis, medical image analysis, and patient outcome prediction with physician-level accuracy. It processes medical data faster than humans but raises concerns about privacy, bias, and regulation."
    },
    {
        "prompt": "Summarize the following text in 2-3 sentences:\n\nClimate change poses one of the greatest challenges of the 21st century. Rising global temperatures driven by greenhouse gas emissions are causing more frequent extreme weather events, rising sea levels, and disruption of ecosystems worldwide. Scientists warn that without immediate and drastic reduction in carbon emissions, the consequences could be catastrophic for both human civilization and biodiversity. Renewable energy sources like solar and wind power are expanding rapidly, but the transition away from fossil fuels requires coordinated global policy action and significant economic investment.\n\nSummary:",
        "reference": "Climate change from greenhouse gas emissions is causing extreme weather, rising seas, and ecosystem disruption. Scientists warn of catastrophic consequences without carbon emission reductions, requiring global policy coordination and investment in renewable energy."
    },
    {
        "prompt": "Summarize the following text in 2-3 sentences:\n\nThe Internet of Things (IoT) refers to the network of physical devices embedded with sensors, software, and connectivity that enables them to collect and exchange data. From smart home devices like thermostats and security cameras to industrial sensors monitoring factory equipment, IoT technology is becoming pervasive. The global IoT market is projected to reach trillions of dollars, with billions of connected devices expected by 2030. Security vulnerabilities in IoT devices remain a significant concern, as poorly secured devices can serve as entry points for cyberattacks on critical infrastructure.\n\nSummary:",
        "reference": "IoT connects physical devices with sensors and software to collect and exchange data, spanning from smart homes to industrial applications. Despite a multi-trillion dollar market projection, IoT security vulnerabilities pose significant risks as entry points for cyberattacks."
    },
    {
        "prompt": "Summarize the following text in 2-3 sentences:\n\nBlockchain technology, originally developed as the foundation for Bitcoin, is a distributed ledger system that records transactions across multiple computers in a way that makes them tamper-resistant. Beyond cryptocurrency, blockchain applications now extend to supply chain management, digital identity verification, smart contracts, and decentralized finance. The technology offers transparency and security advantages but faces challenges including high energy consumption, scalability limitations, and regulatory uncertainty. Many enterprises are exploring private or permissioned blockchain networks that offer the benefits of the technology while addressing some of its limitations.\n\nSummary:",
        "reference": "Blockchain is a distributed tamper-resistant ledger originally behind Bitcoin, now applied to supply chains, smart contracts, and decentralized finance. Despite transparency and security advantages, it faces challenges in energy use, scalability, and regulation."
    },
    {
        "prompt": "Summarize the following text in 2-3 sentences:\n\nQuantum computing leverages quantum mechanical phenomena such as superposition and entanglement to perform computations that would be intractable for classical computers. While current quantum computers are error-prone and require extreme cooling near absolute zero, they have already demonstrated quantum advantage on specific tasks. Major technology companies including IBM, Google, and Microsoft are investing billions in quantum research. Practical quantum computers capable of breaking current encryption standards or simulating complex molecular systems for drug discovery could be a decade or more away, but the potential impact is transformative.\n\nSummary:",
        "reference": "Quantum computing uses quantum mechanics to perform computations impossible for classical computers, with major tech companies investing billions in its development. Practical quantum computers that could break encryption or aid drug discovery may still be a decade away but promise transformative impact."
    }
]

TRANSLATION_INPUTS = [
    {
        "prompt": "Translate the following English text to French:\n\nThe weather today is sunny and warm. I am going to the park with my friends.\n\nFrench translation:",
        "reference": "Le temps aujourd'hui est ensoleillé et chaud. Je vais au parc avec mes amis."
    },
    {
        "prompt": "Translate the following English text to French:\n\nArtificial intelligence is changing the world rapidly. Many industries are adopting new technologies.\n\nFrench translation:",
        "reference": "L'intelligence artificielle change le monde rapidement. De nombreuses industries adoptent de nouvelles technologies."
    },
    {
        "prompt": "Translate the following English text to French:\n\nThe university library opens at eight in the morning and closes at ten in the evening.\n\nFrench translation:",
        "reference": "La bibliothèque universitaire ouvre à huit heures du matin et ferme à dix heures du soir."
    },
    {
        "prompt": "Translate the following English text to French:\n\nCloud computing allows businesses to store and process data on remote servers instead of local hardware.\n\nFrench translation:",
        "reference": "L'informatique en nuage permet aux entreprises de stocker et de traiter des données sur des serveurs distants plutôt que sur du matériel local."
    },
    {
        "prompt": "Translate the following English text to French:\n\nMachine learning models require large amounts of data to achieve high accuracy on complex tasks.\n\nFrench translation:",
        "reference": "Les modèles d'apprentissage automatique nécessitent de grandes quantités de données pour atteindre une haute précision sur des tâches complexes."
    }
]

REASONING_INPUTS = [
    {
        "prompt": "Answer the following question with a number only:\n\nIf a train travels at 60 km/h and needs to cover 180 km, how many hours will the journey take?\n\nAnswer:",
        "reference": "3"
    },
    {
        "prompt": "Answer the following question with a number only:\n\nA store sells apples at 3 for $1. How much do 12 apples cost in dollars?\n\nAnswer:",
        "reference": "4"
    },
    {
        "prompt": "Answer the following logic question:\n\nAll cats are animals. All animals need food. Does a cat need food? Answer yes or no.\n\nAnswer:",
        "reference": "yes"
    },
    {
        "prompt": "Answer the following question with a number only:\n\nWhat is 15% of 200?\n\nAnswer:",
        "reference": "30"
    },
    {
        "prompt": "Answer the following logic question:\n\nIf it rains, the ground gets wet. The ground is not wet. Did it rain? Answer yes or no.\n\nAnswer:",
        "reference": "no"
    }
]

TASKS = {
    "summarization": SUMMARIZATION_INPUTS,
    "translation": TRANSLATION_INPUTS,
    "reasoning": REASONING_INPUTS
}

print(f"Tasks loaded: {list(TASKS.keys())}")
print(f"Total benchmark runs: {sum(len(v) for v in TASKS.values())} examples × 3 models = {sum(len(v) for v in TASKS.values()) * 3} inference calls")

In [ ]:
# Cell 5 — Model inference function
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

def run_model(model_id: str, prompt: str, max_new_tokens: int = 150):
    """Load model, run inference, return (response_text, latency_s, memory_mb, tokens_per_sec)"""
    print(f"  Loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    model.eval()

    torch.cuda.reset_peak_memory_stats()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    input_len = inputs["input_ids"].shape[1]

    t0 = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = round(time.time() - t0, 3)

    mem_mb = round(torch.cuda.max_memory_allocated() / 1e6, 1)
    tokens_generated = output.shape[1] - input_len
    tps = round(tokens_generated / latency, 2) if latency > 0 else 0

    response = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    del model
    torch.cuda.empty_cache()

    return response, latency, mem_mb, tps

print("Inference function ready")

In [ ]:
# Cell 6 — Metrics functions
from evaluate import load as load_metric
import sacrebleu as sb

rouge_metric = load_metric("rouge")

def compute_rouge(hypothesis: str, reference: str) -> dict:
    result = rouge_metric.compute(predictions=[hypothesis], references=[reference])
    return {
        "rouge1": round(result["rouge1"], 4),
        "rouge2": round(result["rouge2"], 4),
        "rougeL": round(result["rougeL"], 4)
    }

def compute_bleu(hypothesis: str, reference: str) -> float:
    result = sb.corpus_bleu([hypothesis], [[reference]])
    return round(result.score, 4)

def compute_accuracy(response: str, expected: str) -> float:
    return 1.0 if expected.lower().strip() in response.lower().strip() else 0.0

print("Metrics functions ready")

In [ ]:
# Cell 7 — Main benchmark loop with W&B logging
import pandas as pd
import wandb

MODELS = {
    "LLaMA-3.2-3B": "meta-llama/Llama-3.2-3B-Instruct",
    "Falcon-7B":    "tiiuae/falcon-7b-instruct",
    "Mistral-7B":   "mistralai/Mistral-7B-Instruct-v0.2"
}

wandb.init(
    project="llm-benchmarking",
    entity="rvce_abhi-potharaju",
    config={"models": list(MODELS.keys()), "tasks": list(TASKS.keys())}
)

results = []

for model_name, model_id in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Benchmarking: {model_name}")
    print(f"{'='*60}")

    for task_name, examples in TASKS.items():
        print(f"\n  Task: {task_name}")

        for i, example in enumerate(examples):
            print(f"    Example {i+1}/5...", end=" ")

            try:
                response, latency, mem_mb, tps = run_model(model_id, example["prompt"])

                row = {
                    "model": model_name,
                    "task": task_name,
                    "example_id": i + 1,
                    "response": response[:200],
                    "latency_s": latency,
                    "memory_mb": mem_mb,
                    "tokens_per_sec": tps,
                    "rouge1": None, "rouge2": None, "rougeL": None,
                    "bleu": None,
                    "accuracy": None
                }

                if task_name == "summarization":
                    scores = compute_rouge(response, example["reference"])
                    row.update(scores)
                    print(f"ROUGE-L={scores['rougeL']:.3f} | latency={latency}s | mem={mem_mb}MB")

                elif task_name == "translation":
                    row["bleu"] = compute_bleu(response, example["reference"])
                    print(f"BLEU={row['bleu']:.2f} | latency={latency}s | mem={mem_mb}MB")

                elif task_name == "reasoning":
                    row["accuracy"] = compute_accuracy(response, example["reference"])
                    print(f"Accuracy={row['accuracy']} | latency={latency}s | mem={mem_mb}MB")

                results.append(row)
                wandb.log({k: v for k, v in row.items() if v is not None and k not in ["response"]})

            except Exception as e:
                print(f"ERROR: {e}")
                results.append({"model": model_name, "task": task_name, "example_id": i+1, "error": str(e)})

wandb.finish()
df = pd.DataFrame(results)
print(f"\nBenchmark complete! {len(df)} results collected.")
df.head()

In [ ]:
# Cell 8 — View summary statistics
import pandas as pd

print("=== RESULTS SUMMARY ===")
print()

summary = df.groupby(["model", "task"]).agg({
    "latency_s": "mean",
    "memory_mb": "mean",
    "tokens_per_sec": "mean",
    "rouge1": "mean",
    "rouge2": "mean",
    "rougeL": "mean",
    "bleu": "mean",
    "accuracy": "mean"
}).round(4)

print(summary.to_string())

In [ ]:
# Cell 9 — Push results to HuggingFace Datasets Hub
from datasets import Dataset

clean_df = df.fillna(0)
dataset = Dataset.from_pandas(clean_df, preserve_index=False)

dataset.push_to_hub(
    "abhinavp10/llm-benchmark-results",
    token=HF_TOKEN,
    commit_message="Benchmark run: LLaMA-3.2-3B vs Falcon-7B vs Mistral-7B"
)

print("Results saved to HuggingFace Datasets Hub!")
print("Dataset URL: https://huggingface.co/datasets/abhinavp10/llm-benchmark-results")
print()
print("Dashboard URL: https://huggingface.co/spaces/abhinavp10/llm-benchmarking")
print("W&B Dashboard: https://wandb.ai/rvce_abhi-potharaju/llm-benchmarking")